# Portfolio Construction

This notebook transforms the quantitative screener candidate pool into a
provisional portfolio for the Reto Actinver.

The construction process has two stages:

1. Select the highest-ranked candidates within each asset category.
2. Allocate capital within each category using a score-to-volatility rule.

The initial category allocation is:

$$
60\% \text{ STOCK}, \qquad
30\% \text{ ETF}, \qquad
10\% \text{ FUND}.
$$

Within each category, preliminary weights are proportional to:

$$
\tilde{w}_i =
\frac{\max(Score_i,0)}{\sigma_i}.
$$

Final weights are normalized and constrained by a maximum weight per asset.

These rules are provisional and will later be evaluated through historical
backtesting.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Project root
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"

CANDIDATE_POOL_PATH = (
    DATA_DIR / "candidate_pool.csv"
)

if not CANDIDATE_POOL_PATH.exists():
    raise FileNotFoundError(
        "candidate_pool.csv not found. "
        "Run 02_quant_screener.ipynb first."
    )

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\alexi\OneDrive\Desktop\ProyectosPRO\bmv-quant-portfolio-system


In [2]:
candidate_pool = pd.read_csv(
    CANDIDATE_POOL_PATH,
    parse_dates=["date"]
)

print(
    f"Candidate assets: "
    f"{len(candidate_pool)}"
)

display(
    candidate_pool[
        [
            "actinver_symbol",
            "asset_type",
            "score",
            "category_rank",
            "category_percentile",
            "volatility_20d_ann"
        ]
    ]
)

Candidate assets: 21


,actinver_symbol,asset_type,score,category_rank,category_percentile,volatility_20d_ann
0,VGT,ETF,0.723934,1.0,1.000000,0.145041
1,QQQ,ETF,0.598058,2.0,0.980000,0.136006
2,EWZ,ETF,0.590830,3.0,0.960000,0.191857
3,SHV,ETF,0.573330,4.0,0.940000,0.094412
4,SPYM,ETF,0.563161,5.0,0.920000,0.109454
5,VTI,ETF,0.558154,6.0,0.900000,0.111788
6,ACTI500B,FUND,0.371331,1.0,1.000000,0.088935
7,ACTIRVTB,FUND,0.311286,2.0,0.937500,0.042241
8,TX,STOCK,0.934114,1.0,1.000000,0.208347
9,TSMN,STOCK,0.914934,2.0,0.992126,0.276782


In [3]:
# ---------------------------------------------------------
# Portfolio configuration
# ---------------------------------------------------------

ASSET_COUNTS = {
    "STOCK": 6,
    "ETF": 3,
    "FUND": 1
}

CATEGORY_WEIGHTS = {
    "STOCK": 0.60,
    "ETF": 0.30,
    "FUND": 0.10
}

MAX_ASSET_WEIGHT = 0.15

## 1. Final candidate selection

The screener already selects the highest-ranked percentile within each asset
category.

For the provisional portfolio, the highest-ranked instruments are selected
within each category:

- 6 stocks
- 3 ETFs
- 1 fund

This produces a 10-asset portfolio that is easier to manage during the
competition while maintaining diversification across asset types.

In [5]:
selected_assets = []

for asset_type, n_assets in ASSET_COUNTS.items():

    category = (
        candidate_pool[
            candidate_pool["asset_type"]
            == asset_type
        ]
        .sort_values(
            "category_rank"
        )
        .head(n_assets)
        .copy()
    )

    selected_assets.append(
        category
    )

portfolio = pd.concat(
    selected_assets,
    ignore_index=True
)

portfolio[
    [
        "actinver_symbol",
        "asset_type",
        "score",
        "category_rank",
        "volatility_20d_ann"
    ]
]

,actinver_symbol,asset_type,score,category_rank,volatility_20d_ann
0,TX,STOCK,0.934114,1.0,0.208347
1,TSMN,STOCK,0.914934,2.0,0.276782
2,NVDA,STOCK,0.824437,3.0,0.265913
3,TGT,STOCK,0.812531,4.0,0.237067
4,MSFT,STOCK,0.796523,5.0,0.225412
5,FIBRAMQ12,STOCK,0.792100,6.0,0.115674
6,VGT,ETF,0.723934,1.0,0.145041
7,QQQ,ETF,0.598058,2.0,0.136006
8,EWZ,ETF,0.590830,3.0,0.191857
9,ACTI500B,FUND,0.371331,1.0,0.088935


## 2. Risk-adjusted weighting

Within each asset category, the preliminary allocation is proportional to the
asset's quantitative score divided by its annualized volatility:

$$
\tilde{w}_i =
\frac{\max(Score_i,0)}
{\sigma_i}.
$$

Therefore:

- a higher quantitative score increases the allocation;
- higher volatility reduces the allocation.

The raw weights are calculated separately within each asset category because
the screener factors were standardized within category.

In [6]:
portfolio["positive_score"] = (
    portfolio["score"]
    .clip(lower=0)
)

portfolio["weight_signal"] = (
    portfolio["positive_score"]
    / portfolio["volatility_20d_ann"]
)

In [8]:
portfolio["within_category_weight"] = (
    portfolio
    .groupby("asset_type")["weight_signal"]
    .transform(
        lambda x: x / x.sum()
    )
)

In [9]:
portfolio["category_weight"] = (
    portfolio["asset_type"]
    .map(CATEGORY_WEIGHTS)
)

portfolio["preliminary_weight"] = (
    portfolio["within_category_weight"]
    * portfolio["category_weight"]
)

In [12]:
def apply_weight_cap(
    weights,
    max_weight,
    target_total
):

    weights = weights.copy()

    if len(weights) * max_weight < target_total:
        raise ValueError(
            "Weight cap is too restrictive "
            "for the number of assets."
        )

    weights = (
        weights
        / weights.sum()
        * target_total
    )

    while (
        weights > max_weight
    ).any():

        capped = (
            weights >= max_weight
        )

        weights.loc[capped] = (
            max_weight
        )

        remaining_total = (
            target_total
            - weights.loc[capped].sum()
        )

        uncapped = ~capped

        if not uncapped.any():
            break

        uncapped_sum = (
            weights.loc[uncapped]
            .sum()
        )

        weights.loc[uncapped] = (
            weights.loc[uncapped]
            / uncapped_sum
            * remaining_total
        )

    return weights

In [13]:
portfolio["weight"] = 0.0

for asset_type, category_weight in (
    CATEGORY_WEIGHTS.items()
):

    mask = (
        portfolio["asset_type"]
        == asset_type
    )

    category_weights = (
        portfolio.loc[
            mask,
            "preliminary_weight"
        ]
    )

    portfolio.loc[
        mask,
        "weight"
    ] = apply_weight_cap(
        weights=category_weights,
        max_weight=MAX_ASSET_WEIGHT,
        target_total=category_weight
    )

In [14]:
assert np.isclose(
    portfolio["weight"].sum(),
    1.0
)

assert (
    portfolio["weight"]
    <= MAX_ASSET_WEIGHT + 1e-10
).all()

for asset_type, target in (
    CATEGORY_WEIGHTS.items()
):

    actual = (
        portfolio.loc[
            portfolio["asset_type"]
            == asset_type,
            "weight"
        ].sum()
    )

    assert np.isclose(
        actual,
        target
    )

print(
    "Portfolio validation successful."
)

Portfolio validation successful.


In [15]:
final_portfolio = (
    portfolio[
        [
            "actinver_symbol",
            "asset_type",
            "score",
            "category_rank",
            "volatility_20d_ann",
            "current_drawdown",
            "weight"
        ]
    ]
    .sort_values(
        "weight",
        ascending=False
    )
    .copy()
)

final_portfolio["weight_pct"] = (
    final_portfolio["weight"]
    * 100
)

display(
    final_portfolio[
        [
            "actinver_symbol",
            "asset_type",
            "score",
            "volatility_20d_ann",
            "current_drawdown",
            "weight_pct"
        ]
    ]
)

,actinver_symbol,asset_type,score,volatility_20d_ann,current_drawdown,weight_pct
5,FIBRAMQ12,STOCK,0.792100,0.115674,-0.002062,15.000000
6,VGT,ETF,0.723934,0.145041,0.000000,12.009641
0,TX,STOCK,0.934114,0.208347,0.000000,11.302490
7,QQQ,ETF,0.598058,0.136006,0.000000,10.580518
9,ACTI500B,FUND,0.371331,0.088935,-0.001193,10.000000
4,MSFT,STOCK,0.796523,0.225412,-0.064945,8.908041
3,TGT,STOCK,0.812531,0.237067,0.000000,8.640328
1,TSMN,STOCK,0.914934,0.276782,0.000000,8.333247
2,NVDA,STOCK,0.824437,0.265913,0.000000,7.815894
8,EWZ,ETF,0.590830,0.191857,-0.026836,7.409841


In [16]:
category_allocation = (
    final_portfolio
    .groupby("asset_type")
    .agg(
        assets=(
            "actinver_symbol",
            "count"
        ),
        weight=(
            "weight",
            "sum"
        )
    )
)

category_allocation[
    "weight_pct"
] = (
    category_allocation["weight"]
    * 100
)

category_allocation

,assets,weight,weight_pct
asset_type,,,
ETF,3,0.3,30.0
FUND,1,0.1,10.0
STOCK,6,0.6,60.0


In [17]:
OUTPUT_PATH = (
    DATA_DIR
    / "current_portfolio.csv"
)

final_portfolio.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"Portfolio saved to: "
    f"{OUTPUT_PATH}"
)

Portfolio saved to: c:\Users\alexi\OneDrive\Desktop\ProyectosPRO\bmv-quant-portfolio-system\data\processed\current_portfolio.csv
